# Notebook 02 - Preprocessing & Feature Engineering
**Projet : SentimentsLoupe - Phase 1 Flipkart**

---
## Objectifs
1. Appliquer le pipeline de nettoyage texte (TextCleaner)
2. Analyser l'impact du nettoyage (avant / après)
3. Construire les features TF-IDF + features numériques
4. Analyser le vocabulaire TF-IDF (top features par classe)
5. Sauvegarder le dataset prêt pour l'entraînement

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from scipy.sparse import issparse

from src.config.settings import (
    FLIPKART_RAW, FLIPKART_CLEAN, MODELS_DIR, PLOTS_DIR,
    RANDOM_STATE, TEST_SIZE, SENTIMENT_MAP, LABEL2ID
)
from src.preprocessing.text_cleaner import TextCleaner
from src.preprocessing.feature_engineering import FeatureEngineer

import os
os.makedirs(str(PLOTS_DIR), exist_ok=True)
os.makedirs(str(MODELS_DIR), exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
PALETTE = {'positive': '#1D9E75', 'neutral': '#888780', 'negative': '#E24B4A'}
print('Setup OK')

## 1. Chargement & Suppression des Doublons

In [ ]:
df_raw = pd.read_csv(str(FLIPKART_RAW))
print(f'Shape brut   : {df_raw.shape}')

# Supprimer les doublons AVANT le split (fuite de données si après)
df = df_raw.drop_duplicates(subset=['review']).reset_index(drop=True)
print(f'Après dedup  : {df.shape} ({df_raw.shape[0] - df.shape[0]} doublons supprimés)')

# Distribution après dedup
df['sentiment'] = df['rating'].map(SENTIMENT_MAP)
df['label']     = df['sentiment'].map(LABEL2ID)
print(f'\nDistribution après dedup:')
print(df['sentiment'].value_counts())

# Est-ce que le déséquilibre s'est empiré?
pos = (df['sentiment']=='positive').sum()
neg = (df['sentiment']=='negative').sum()
print(f'\nRatio pos/neg après dedup: {pos/neg:.1f}x')

## 2. Pipeline de Nettoyage - Démonstration

In [ ]:
# Démonstration sur des exemples représentatifs
examples = [
    "It was nice produt. I like it's design a lot. READ MORE",
    "awesome sound....very pretty to see this nd the sound quality was too good I wish to take this product loved this product \U0001f60d\U0001f60d\U0001f60dREAD MORE",
    "WORST PRODUCT EVER!!! Complete WASTE of MONEY. DO NOT BUY!!!",
    "ok product.. works fine READ MORE",
    "sooooo good I looooved it veryyy much!",
    "Check https://example.com for better deals",
]

cleaner = TextCleaner(lemmatize=True, remove_stopwords=False, min_word_len=2)

print('=' * 80)
print(f'{"BRUT":^38} | {"NETTOYÉ":^38}')
print('=' * 80)
for ex in examples:
    cleaned = cleaner.clean(ex)
    raw_disp     = ex[:37].ljust(38)
    cleaned_disp = cleaned[:37].ljust(38)
    print(f'{raw_disp} | {cleaned_disp}')
print('=' * 80)

In [ ]:
# Appliquer sur tout le dataset
df_clean = cleaner.fit_transform(df, col='review')
print(f'\nShape après nettoyage : {df_clean.shape}')
print(f'Colonnes ajoutées    : {[c for c in df_clean.columns if c not in df.columns]}')

## 3. Analyse Avant / Après Nettoyage

In [ ]:
# Comparaison longueur avant/après
df_clean['word_count_raw']   = df_clean['review'].str.split().str.len()
df_clean['word_count_clean'] = df_clean['review_clean'].str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Impact du Nettoyage sur le Texte', fontsize=13, fontweight='bold')

axes[0].hist(df_clean['word_count_raw'],   bins=40, color='#E24B4A', alpha=0.7, label='Brut',    density=True)
axes[0].hist(df_clean['word_count_clean'], bins=40, color='#1D9E75', alpha=0.7, label='Nettoyé', density=True)
axes[0].set_title('Distribution longueur (avant/après)', fontweight='bold')
axes[0].set_xlabel('Nombre de mots')
axes[0].legend()

sample = df_clean.sample(500, random_state=42)
axes[1].scatter(sample['word_count_raw'], sample['word_count_clean'],
                alpha=0.3, s=10,
                c=[{'positive':'#1D9E75','neutral':'#888780','negative':'#E24B4A'}[s] for s in sample['sentiment']])
axes[1].plot([0, 100], [0, 100], 'k--', alpha=0.4, label='y=x (pas de changement)')
axes[1].set_title('Longueur brut vs nettoyé (500 échantillons)', fontweight='bold')
axes[1].set_xlabel('Mots bruts')
axes[1].set_ylabel('Mots après nettoyage')
axes[1].legend(fontsize=8)

reduction = {
    'Mots bruts': df_clean['word_count_raw'].mean(),
    'Mots nettoyés': df_clean['word_count_clean'].mean()
}
bars = axes[2].bar(reduction.keys(), reduction.values(),
                   color=['#E24B4A', '#1D9E75'], width=0.5, edgecolor='white')
axes[2].set_title('Longueur moyenne avant/après', fontweight='bold')
axes[2].set_ylabel('Mots moyens par avis')
for bar, v in zip(bars, reduction.values()):
    axes[2].text(bar.get_x() + bar.get_width()/2, v + 0.1,
                 f'{v:.1f}', ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig(str(PLOTS_DIR / '02a_cleaning_impact.png'), bbox_inches='tight', dpi=150)
plt.show()

print(f'Reduction vocabulaire : {df_clean["word_count_raw"].mean():.1f} -> {df_clean["word_count_clean"].mean():.1f} mots/avis')

## 4. Feature Engineering - TF-IDF

In [ ]:
# Split stratifié (AVANT de fitter le TF-IDF - prévention fuite de données)
X_train_df, X_test_df, y_train, y_test = train_test_split(
    df_clean,
    df_clean['label'],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df_clean['label']
)

print(f'Train : {len(X_train_df)} avis')
print(f'Test  : {len(X_test_df)} avis')
print(f'\nDistribution train (labels):')
print(y_train.value_counts().sort_index())
print(f'\nDistribution test (labels):')
print(y_test.value_counts().sort_index())

train_ratio = y_train.value_counts(normalize=True).sort_index()
test_ratio  = y_test.value_counts(normalize=True).sort_index()
print(f'\nRatio train vs test (doit etre proche):')
for label in [0, 1, 2]:
    print(f'  Classe {label}: train={train_ratio.get(label, 0):.3f} | test={test_ratio.get(label, 0):.3f}')

In [ ]:
# Feature Engineering
fe = FeatureEngineer(use_numeric_features=True)

X_train = fe.fit_transform(X_train_df)
X_test  = fe.transform(X_test_df)

print(f'\nShape features:')
print(f'  X_train : {X_train.shape}')
print(f'  X_test  : {X_test.shape}')
print(f'  Sparse  : {issparse(X_train)}')
print(f'  Densité : {X_train.nnz / (X_train.shape[0] * X_train.shape[1]) * 100:.4f}%')
print(f'  Taille mémoire sparse : ~{X_train.data.nbytes / 1024 / 1024:.1f} MB')

## 5. Analyse des Features TF-IDF

In [ ]:
# Top features TF-IDF par classe
from sklearn.feature_extraction.text import TfidfVectorizer

# Re-fit TF-IDF uniquement sur le texte pour l'analyse (pas les features numériques)
tfidf_analysis = TfidfVectorizer(
    max_features=15000, ngram_range=(1, 2), min_df=2,
    sublinear_tf=True, strip_accents='unicode'
)
X_tfidf_full = tfidf_analysis.fit_transform(df_clean['review_clean'])
feature_names = np.array(tfidf_analysis.get_feature_names_out())

print(f'Vocabulaire TF-IDF : {len(feature_names)} features')

# Score moyen TF-IDF par classe (utiliser .values pour éviter l'erreur sparse indexing)
y_full = df_clean['label'].values

def top_tfidf_by_class(X, y, feature_names, class_id, n=15):
    mask = y == class_id
    class_tfidf = np.asarray(X[mask].mean(axis=0)).flatten()
    top_idx = class_tfidf.argsort()[-n:][::-1]
    return list(zip(feature_names[top_idx], class_tfidf[top_idx]))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Top 12 features TF-IDF par classe de sentiment', fontsize=13, fontweight='bold')

for ax, (class_id, name) in enumerate([(0,'negative'),(1,'neutral'),(2,'positive')]):
    top = top_tfidf_by_class(X_tfidf_full, y_full, feature_names, class_id, n=12)
    words, scores = zip(*top)
    ax_ = axes[ax]
    bars = ax_.barh(list(words)[::-1], list(scores)[::-1],
                    color=PALETTE[name], alpha=0.85, edgecolor='white')
    ax_.set_title(f'{name.upper()}', fontweight='bold', color=PALETTE[name], fontsize=12)
    ax_.set_xlabel('Score TF-IDF moyen')
    for bar, score in zip(bars, list(scores)[::-1]):
        ax_.text(score + 0.0001, bar.get_y() + bar.get_height()/2,
                 f'{score:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(str(PLOTS_DIR / '02b_tfidf_features.png'), bbox_inches='tight', dpi=150)
plt.show()

## 6. Impact des Features Numériques

In [ ]:
# Ajouter les signaux lexicaux au df_clean pour l'analyse
df_clean['has_excl'] = df_clean['review'].str.contains('!', regex=False).astype(int)
df_clean['has_quest'] = df_clean['review'].str.contains(r'\?', regex=True).astype(int)
df_clean['caps_ratio'] = df_clean['review'].apply(
    lambda t: sum(1 for c in str(t) if c.isupper()) / max(len(str(t)), 1)
)

num_features = ['word_count', 'char_count', 'avg_word_len', 'has_excl', 'has_quest', 'caps_ratio']
available = [c for c in num_features if c in df_clean.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Features numériques par classe de sentiment', fontsize=13, fontweight='bold')
axes_flat = axes.flatten()

for ax, feat in zip(axes_flat, available):
    for sentiment in ['positive', 'neutral', 'negative']:
        data = df_clean[df_clean['sentiment'] == sentiment][feat]
        if feat in ['has_excl', 'has_quest']:
            pass
        else:
            ax.hist(data, bins=25, alpha=0.55, color=PALETTE[sentiment],
                    label=sentiment, density=True)
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold')
    ax.legend(fontsize=8)

# Remplacer les features binaires par des bar charts
for i, feat in enumerate(['has_excl', 'has_quest']):
    if feat not in available:
        continue
    ax = axes_flat[available.index(feat)]
    ax.cla()
    rates = [df_clean[df_clean['sentiment']==s][feat].mean()*100 for s in ['negative','neutral','positive']]
    bars = ax.bar(['neg','neu','pos'], rates,
                  color=[PALETTE['negative'], PALETTE['neutral'], PALETTE['positive']],
                  edgecolor='white')
    ax.set_title(feat.replace('_', ' ').title() + ' (%)', fontweight='bold')
    ax.set_ylabel('% des avis')
    for bar, v in zip(bars, rates):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.2, f'{v:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(str(PLOTS_DIR / '02c_numeric_features.png'), bbox_inches='tight', dpi=150)
plt.show()

## 7. Sauvegarde

In [ ]:
import joblib

# Dataset nettoyé
df_clean.to_csv(str(FLIPKART_CLEAN), index=False)
print(f'Dataset nettoyé : {FLIPKART_CLEAN}')

# FeatureEngineer (TF-IDF fitté)
fe.save(MODELS_DIR / 'feature_engineer.pkl')
print(f'FeatureEngineer : {MODELS_DIR}/feature_engineer.pkl')

# Sauvegarder les indices du split pour cohérence avec le notebook 03
split_info = {
    'train_indices': X_train_df.index.tolist(),
    'test_indices':  X_test_df.index.tolist()
}
joblib.dump(split_info, str(MODELS_DIR / 'split_indices.pkl'))
print(f'Split indices    : {MODELS_DIR}/split_indices.pkl')

# Sauvegarder X_train, X_test, y_train, y_test
joblib.dump({'X_train': X_train, 'X_test': X_test,
             'y_train': y_train.values, 'y_test': y_test.values},
             str(MODELS_DIR / 'features_ready.pkl'))
print(f'Features prêtes : {MODELS_DIR}/features_ready.pkl')

print(f'\nRésumé features:')
print(f'   X_train : {X_train.shape}')
print(f'   X_test  : {X_test.shape}')
print(f'   Suite : Notebook 03 - Modèles Baseline & SMOTE')